# Averaged Voltage Sampling with Digilent Analog Discovery

📊 **Purpose**:  
This notebook uses the `pydwf` library to configure the **Analog Discovery** device to perform voltage sampling on **Analog Input Channel 1**.

### What it does:
- Takes analog voltage samples for **1 second** at **1000 Hz** (i.e., 1000 samples).
- Repeats this process **16 times** (over 16 seconds).
- For each second's data, it computes the **average voltage**.
- Returns and prints a list of the 16 average voltages.

> Useful for slow-changing signals where per-second averaging is more relevant than high-frequency details.


In [2]:
import time
import numpy as np
from pydwf import DwfLibrary
from pydwf.utilities import openDwfDevice
from pydwf.core.api.analog_in import DwfAcquisitionMode, DwfAnalogInFilter

In [ ]:
# Initialize DWF
dwf = DwfLibrary()
with openDwfDevice(dwf) as device:
    analogIn = device.analogIn

    # Reset and configure AnalogIn
    analogIn.reset()
    analogIn.channelEnableSet(0, True)  # Channel 1 (index 0)
    analogIn.channelFilterSet(0, DwfAnalogInFilter.Average)  # Apply averaging
    analogIn.channelRangeSet(0, 5.0)  # Set voltage range, e.g., ±5V
    analogIn.acquisitionModeSet(DwfAcquisitionMode.Record)
    
    analogIn.frequencySet(1000.0)  # 1000 samples/sec
    analogIn.recordLengthSet(1.0)  # Record for 1 second
    analogIn.bufferSizeSet(8192)   # Use a reasonably large buffer

    averages = []

    for i in range(16):
        analogIn.configure(reconfigure=True, start=True)

        # Wait until acquisition is done
        while True:
            state = analogIn.status(True)
            if state.name == "Done":
                break
            time.sleep(0.1)

        sample_count = analogIn.statusSamplesValid()
        data = analogIn.statusData(0, sample_count)  # Channel 1 (index 0)
        average = float(np.mean(data))
        print(f"Second {i+1}: Avg = {average:.3f} V")
        averages.append(average)

    print("Averages over 16 seconds:", averages)


Markdown Git Repo Test